In [1]:
!pip install wbgapi pandas plotly statsmodels kaleido -q

import wbgapi as wb
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.express as px

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.7 MB/s eta 0:00:00


In [2]:
# EU-27 country codes
europe = [
    'AUT', 'BEL', 'BGR', 'HRV', 'CYP', 'CZE', 'DNK', 'EST',
    'FIN', 'FRA', 'DEU', 'GRC', 'HUN', 'IRL', 'ITA', 'LVA',
    'LTU', 'LUX', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK',
    'SVN', 'ESP', 'SWE'
]

print("Number of countries:", len(europe))

# Sector codes
sectors = {
    'Manufacturing': 'NV.IND.MANF.ZS',
    'Services': 'NV.SRV.TOTL.ZS',
    'Agriculture': 'NV.AGR.TOTL.ZS'
}

# Load sector data
data = {}
for sector, code in sectors.items():
    df = wb.data.DataFrame(code, economy=europe, time=range(2015, 2023))
    data[sector] = df

print("Sector data loaded!")

Number of countries: 27
Sector data loaded!


In [3]:
automation_risk = {
    'Manufacturing': 0.70,
    'Services': 0.52,
    'Agriculture': 0.55
}

rows = []
for sector, code in sectors.items():
    df = data[sector]
    latest = df['YR2021'].dropna()

    for country, value in latest.items():
        rows.append({
            'country': country,
            'sector': sector,
            'gdp_share': value,
            'automation_risk': automation_risk[sector]
        })

df_all = pd.DataFrame(rows)
df_all['weighted_risk'] = df_all['gdp_share'] * df_all['automation_risk']

country_risk = (
    df_all.groupby('country')['weighted_risk']
    .sum()
    .reset_index()
    .rename(columns={'weighted_risk': 'total_risk_score'})
)

print(country_risk.sort_values('total_risk_score', ascending=False))

   country  total_risk_score
14     IRL         53.274888
5      DEU         46.459061
4      CZE         46.366671
19     MLT         46.231355
15     ITA         45.479034
0      AUT         44.951715
17     LUX         44.933048
25     SVN         44.864072
1      BEL         44.670893
16     LTU         44.640678
20     NLD         44.548896
23     ROU         44.488787
7      ESP         44.478355
6      DNK         44.035369
24     SVK         43.933722
10     FRA         43.922175
11     GRC         43.853545
22     PRT         43.734904
3      CYP         43.728582
26     SWE         43.414612
18     LVA         43.352345
13     HUN         43.301687
9      FIN         42.928370
8      EST         42.803336
21     POL         42.700604
12     HRV         40.714259
2      BGR         34.843683


In [4]:
# Unemployment
unemp = wb.data.DataFrame('SL.UEM.TOTL.ZS', economy=europe, time=range(2021, 2022))
unemp_2021 = unemp['SL.UEM.TOTL.ZS']

# Controls
gdp = wb.data.DataFrame('NY.GDP.PCAP.CD', economy=europe, time=range(2021, 2022))
gdp_2021 = gdp['NY.GDP.PCAP.CD']

edu = wb.data.DataFrame('SE.TER.ENRR', economy=europe, time=range(2021, 2022))
edu_2021 = edu['SE.TER.ENRR']

manuf = wb.data.DataFrame('NV.IND.MANF.ZS', economy=europe, time=range(2021, 2022))
manuf_2021 = manuf['NV.IND.MANF.ZS']

print("All data loaded!")

All data loaded!


In [5]:
df = country_risk.copy()
df['unemployment'] = df['country'].map(unemp_2021)
df['gdp_per_capita'] = df['country'].map(gdp_2021)
df['tertiary_edu'] = df['country'].map(edu_2021)
df['manufacturing'] = df['country'].map(manuf_2021)

df = df.dropna()
df['log_gdp'] = np.log(df['gdp_per_capita'])

print("Final sample size:", len(df))
print(df[['country', 'total_risk_score', 'unemployment', 'log_gdp', 'tertiary_edu', 'manufacturing']].round(2))

Final sample size: 26
   country  total_risk_score  unemployment  log_gdp  tertiary_edu  \
0      AUT             44.95          6.46    10.89         93.94   
1      BEL             44.67          6.25    10.85         82.69   
3      CYP             43.73          7.51    10.40        106.36   
4      CZE             46.37          2.80    10.23         69.84   
5      DEU             46.46          3.59    10.87         75.67   
6      DNK             44.04          5.07    11.15         83.98   
7      ESP             44.48         14.91    10.34         95.38   
8      EST             42.80          6.18    10.24         73.15   
9      FIN             42.93          7.62    10.88        100.87   
10     FRA             43.92          7.87    10.69         69.94   
11     GRC             43.85         14.66     9.94        150.61   
12     HRV             40.71          7.48     9.79         78.01   
13     HUN             43.30          4.02     9.85         58.65   
14     IRL  

In [6]:
y = df['unemployment']

# Model 1: No controls
X1 = sm.add_constant(df['total_risk_score'])
model1 = sm.OLS(y, X1).fit()

# Model 2: With all controls (including manufacturing)
X2 = sm.add_constant(df[['total_risk_score', 'log_gdp', 'tertiary_edu', 'manufacturing']])
model2 = sm.OLS(y, X2).fit()

# Model 3: Without manufacturing (to reduce multicollinearity)
X3 = sm.add_constant(df[['total_risk_score', 'log_gdp', 'tertiary_edu']])
model3 = sm.OLS(y, X3).fit()

print("="*60)
print("MODEL 1: No Controls")
print("="*60)
print(model1.summary())

print("\n" + "="*60)
print("MODEL 2: Full Controls (with Manufacturing)")
print("="*60)
print(model2.summary())

print("\n" + "="*60)
print("MODEL 3: Controls without Manufacturing")
print("="*60)
print(model3.summary())

MODEL 1: No Controls
                            OLS Regression Results                            
Dep. Variable:           unemployment   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                 -0.019
Method:                 Least Squares   F-statistic:                    0.5321
Date:                Wed, 26 Aug 2026   Prob (F-statistic):              0.473
Time:                        09:44:54   Log-Likelihood:                -64.112
No. Observations:                  26   AIC:                             132.2
Df Residuals:                      24   BIC:                             134.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const              

In [7]:
fig = px.scatter(
    df,
    x='total_risk_score',
    y='unemployment',
    text='country',
    title='AI Risk Score vs Unemployment (EU countries, 2021)',
    labels={
        'total_risk_score': 'AI Automation Risk Score',
        'unemployment': 'Unemployment Rate (%)'
    }
)

fig.update_traces(textposition='top center', marker=dict(size=10))
fig.update_layout(height=600)
fig.show()

/usr/local/lib/python3.13/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.


